# Temperature

`temperature` (0.0 to 1.0, default 1.0) controls how random the output is.

- **low (0.0)**: Claude almost always picks the most likely tokens, so answers repeat.
  Use it for extraction, classification, code.
- **high (1.0)**: more varied and creative answers. Use it for brainstorming and writing.
- 0.0 is close to deterministic, not fully guaranteed

## Setup

The client reads `ANTHROPIC_API_KEY` (and `ANTHROPIC_BASE_URL`, if set) from `.env` in the repo root.

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helpers: `chat()` gets `temperature`

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

## Check that temperature changes anything

Ask an open question several times at each temperature and count unique answers.
A question with one right answer ("2 + 2") would look the same at any temperature.

In [3]:
RUNS = 5
PROMPT = "Invent a name for a coffee shop. Reply with the name only."

for temperature in (0.0, 1.0):
    answers = []
    for _ in range(RUNS):
        messages = []
        add_user_message(messages, PROMPT)
        answers.append(chat(messages, temperature=temperature).strip())

    print(f"temperature={temperature}: {len(set(answers))} unique of {RUNS}")
    for answer in answers:
        print("  ", answer)

temperature=0.0: 1 unique of 5
   The Grind House
   The Grind House
   The Grind House
   The Grind House
   The Grind House


temperature=1.0: 4 unique of 5
   The Grind & Gather
   The Brewed Compass
   The Daily Grind House
   The Daily Grind House
   The Cozy Grounds
